### **Dataset Handling**

In [1]:
from typing import Any, Callable
from pathlib import Path
from PIL import Image

from tqdm import tqdm
import torch
import torch.nn as nn
from torch.types import Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as ts
from torchvision.transforms import Compose

import spark as sp

In [2]:
from spark.handle import get_data_filespaths

class ImageDataset(Dataset):
    """
    Baseline AutoEncoder dataset configurator.
    """
    def __init__(self, data_path: str | Path, transform: Compose | None) -> None:
        files_list = get_data_filespaths(data_path, shuffle=True)
        self.data = sp.process_data(files_list, lambda img: Image.open(img).convert('L'), transform)
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, index) -> tuple[Tensor, Tensor]:
        sample = self.data[index]
        return sample, sample

In [3]:
from spark.processing import center_tensor, normalise

#BASEPATH: str = '/home/edoardo/Desktop/MockDataForDMs'
BASEPATH: str = '/mnt/d/MockDataForDMs'
DATASET_ID: str = 'mockPolyImgsDataset'
BATCH_SIZE: int = 250
VALID_SIZE: float = 0.2
PREPROCESSING: Compose = ts.Compose(
    [
        ts.PILToTensor(),
        center_tensor,
        normalise,
    ]
)

In [4]:
def handle_dataset(*filepaths: str | Path, **kwargs) -> tuple[DataLoader, DataLoader | None]:
    """Handles dataset(s). A dataset is loaded if its file exists, else is generated and saved."""
    raise NotImplementedError


try:
    dataset: Dataset = sp.load_dataset(f'{BASEPATH}/{DATASET_ID}.pt')
except FileNotFoundError:
    print('Dataset not found, generating...')
    dataset: ImageDataset = ImageDataset(f'{BASEPATH}/ImgsMockDatasetDMs', PREPROCESSING)
    sp.save_dataset(dataset, f'{BASEPATH}/{DATASET_ID}.pt')

train_dl, valid_dl = sp.get_dataloaders(dataset, BATCH_SIZE, VALID_SIZE)

Loading dataset...
Dataset loaded!
Baking DataLoaders...
DataLoaders ready-to-go!


In [5]:
# import sys

# def get_size(obj: object, default: Any = -1) -> int:
#     """Computes how much memory storage is being used by input obj in [bytes]."""
#     return sys.getsizeof(obj, default=default)

# idx = 0
# for batch in train_dl:
#     # print(type(batch), len(batch))
#     in_data, target = batch
#     print(in_data.shape, torch.allclose(in_data, target), get_size(in_data))
#     idx += 1
#     if idx > 3:
#         break

### **Model Definition**

In [6]:
def get_conv_block(
    in_dims: int,
    out_dims: int,
    kernel_size: int,
    padding: int,
) -> nn.ModuleList:
    """Defines baseline conv block for mock model."""
    block = [
        nn.Conv2d(in_dims, out_dims, kernel_size, padding=padding),
        nn.BatchNorm2d(out_dims),
        nn.ReLU(),
    ]
    return nn.ModuleList(block)


class MockModel(nn.Module):
    """
    Baseline mock CNN model for `spark` API tests.
    """
    def __init__(
        self,
        data_shape: torch.Size,
        out_features: int,
        maxpool: int = 2,
        dropout: float = 0.3, 
    ) -> None:
        super().__init__()
        in_dim = int(data_shape[0])
        in_features = int(data_shape.numel() / pow(maxpool, 2))
        self.net = nn.Sequential(
            *get_conv_block(in_dim, 16, 7, 3),
            *get_conv_block(16, 16, 5, 2),
            *get_conv_block(16, in_dim, 3, 1),
            nn.MaxPool2d(maxpool),
            nn.Flatten(),
            nn.Linear(in_features, 1024), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(1024, out_features),
            nn.Softmax(1),
        )
    
    def forward(self, x: Tensor) -> Tensor:
        out = self.net(x)
        return out

### **Model Handling**

In [7]:
def test_model_working(model: nn.Module, data_shape: tuple[int, int, int]) -> Tensor:
    """Tests if model is working properly when applied."""
    dataset = torch.rand((5, *data_shape))
    out: list[Tensor] = []
    for batch in dataset:
        out.append(model(batch.unsqueeze(0)))
    print("It's alive!!!")
    return out


def save_checkpoint() -> None:
    """Saves a checkpoint during training, given a metric."""
    raise NotImplementedError

In [8]:
MODEL_ID: str = 'mockCNNbasicModel'
DATA_SHAPE: torch.Size = torch.Size([1, 36, 36])
OUT_FEATURES: int = 4

# try:
#     model_data: dict[str, Any] = sp.load_model(f'{BASEPATH}/{MODEL_ID}.pt')
#     model: nn.Module = model_data['state_dict']
# except FileNotFoundError:
#     print('Model not found, generating...')
#     model: nn.Module = MockModel(data_shape=DATA_SHAPE, out_features=OUT_FEATURES)
#     sp.save_model(model, f'{BASEPATH}/{MODEL_ID}.pt')

model: nn.Module = MockModel(data_shape=DATA_SHAPE, out_features=OUT_FEATURES)
print(model)

MockModel(
  (net): Sequential(
    (0): Conv2d(1, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(16, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=324, out_features=1024, bias=True)
    (12): ReLU()
    (13): Dropout(p=0.3, inplace=False)
    (14): Linear(in_features=1024, out_features=4, bias=True)
    (15): Softmax(dim=1)
  )
)


In [9]:
out = test_model_working(model, DATA_SHAPE)

It's alive!!!


### **AutoEncoder Model Handling**

In [10]:
class Normalise(nn.Module):
    def __init__(self, norm_range: str = 'unilateral') -> None:
        super().__init__()
        self.norm_range = norm_range
        return
    
    def __call__(self, tensor: Tensor):
        return normalise(tensor, self.norm_range)


def get_encoder(in_channels: int, latent_dim: int) -> nn.Sequential:
    """Defines baseline encoder for mock AutoEncoder model."""
    arch = [
        nn.Conv2d(in_channels, 32, kernel_size=5, stride=1),
        nn.ReLU(),
        nn.Conv2d(32, 32, kernel_size=5, stride=1),
        nn.ReLU(),
        nn.Conv2d(32, 32, kernel_size=4, stride=2),
        nn.ReLU(),
        nn.Conv2d(32, 32, kernel_size=3, stride=2),
        nn.ReLU(),
        nn.Conv2d(32, latent_dim, kernel_size=4, stride=1),
    ]
    return nn.Sequential(*arch)

def get_decoder(latent_dim: int, out_channels: int) -> nn.Sequential:
    """Defines baseline decoder for mock AutoEncoder model."""
    arch = [
        nn.ConvTranspose2d(latent_dim, 32, kernel_size=4, stride=1),
        nn.ReLU(),
        nn.ConvTranspose2d(32, 32, kernel_size=3, stride=2),
        nn.ReLU(),
        nn.ConvTranspose2d(32, 32, kernel_size=4, stride=2),
        nn.ReLU(),
        nn.ConvTranspose2d(32, 32, kernel_size=5, stride=1),
        nn.ReLU(),
        nn.ConvTranspose2d(32, out_channels, kernel_size=5, stride=1),
        #Normalise(),
    ]
    return nn.Sequential(*arch)


class MockAutoEncoder(nn.Module):
    """
    Baseline mock AutoEncoder model for `spark` API tests and stuff.
    """
    def __init__(self, in_channels: int, latent_dim: int) -> None:
        super().__init__()
        self.encoder = get_encoder(in_channels, latent_dim)
        self.decoder = get_decoder(latent_dim, in_channels)
    
    def forward(self, x: Tensor) -> Tensor:
        embedded = self.encoder(x)
        out = self.decoder(embedded)
        return out

In [11]:
IN_CHANNELS: int = 1
LATENT_DIM: int = 8

autoencoder = MockAutoEncoder(IN_CHANNELS, LATENT_DIM)
print(autoencoder)
# out = test_model_working(autoencoder, DATA_SHAPE)  ---- WORKS!

MockAutoEncoder(
  (encoder): Sequential(
    (0): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2))
    (5): ReLU()
    (6): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2))
    (7): ReLU()
    (8): Conv2d(32, 8, kernel_size=(4, 4), stride=(1, 1))
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(8, 32, kernel_size=(4, 4), stride=(1, 1))
    (1): ReLU()
    (2): ConvTranspose2d(32, 32, kernel_size=(3, 3), stride=(2, 2))
    (3): ReLU()
    (4): ConvTranspose2d(32, 32, kernel_size=(4, 4), stride=(2, 2))
    (5): ReLU()
    (6): ConvTranspose2d(32, 32, kernel_size=(5, 5), stride=(1, 1))
    (7): ReLU()
    (8): ConvTranspose2d(32, 1, kernel_size=(5, 5), stride=(1, 1))
  )
)


In [12]:
import itertools as it
import warnings

import torch.optim as opt
from torch.amp import GradScaler, autocast

def training_setup(
    model: nn.Module,
    loss: Callable,
    optimiser: Callable,
    scheduler: Callable,
    device: str,
) -> dict[str, Callable]:
    """
    Creates a container with training operations.
    NOTE:
        * the scheduler refers to the learning rate.
        * other schedulers for log train/valid loss vals,
          checkpoints, wandb are NOT accounted for as of now  
    """
    print(f'## Operating on device: {device}.')
    setup: dict[str, Any] = {
        'model': model,
        'loss': loss,
        'optimiser': optimiser,
        'scheduler': scheduler,
        'device': device,
    }
    n_params = sum(p.numel() for p in model.parameters())
    print(f'## Model parameters: {n_params}.')
    return setup


def train_model(
    params: dict[str, Any],
    train_dl: DataLoader,
    epochs: int,
    learning_rate: float,
    valid_dl: DataLoader | None = None,
) -> tuple[list[float], list[float]]:
    """
    Basic train routine.
    """
    def check_loss_val(loss_val: float, msg: str) -> bool:
        """Checks the current loss value and raises a warning if is NaN."""
        if not torch.isnan(loss_val):
            return True
        warnings.warn(msg)
        return False

    # config procedure/loss container
    nbatches: int = len(train_dl)
    total_steps: int = epochs * nbatches
    avg_train_loss, avg_valid_loss = [], []

    # setup model/optimiser/scaler for memory saving
    device = params['device']
    model = params['model'].to(device)
    loss_fn = params['loss']
    optimiser = params['optimiser'](model.parameters(), lr=learning_rate)
    scheduler = params['scheduler'](optimiser, patience=5, factor=0.5)
    scaler = GradScaler(device)

    # train model
    train_iter = tqdm(
        iterable=enumerate(it.islice(it.cycle(train_dl), total_steps)),
        desc='Training Model',
    )
    running_batches = 0
    running_train_loss = 0.0

    model.train()
    for idx, (input_batch, trg_batch) in train_iter:
        # update tqdm bar to keep track of the routine
        epoch, batch = divmod(idx, nbatches)
        current_lr = optimiser.param_groups[0]['lr']
        info: dict = {
            'epoch': f'{epoch + 1}/{epochs}',
            'batch': f'{batch + 1}/{nbatches}',
            'lr': f'{current_lr:.2e}'
        }
        train_iter.set_postfix(info)

        # model optimisation
        input_batch, trg_batch = input_batch.to(device), trg_batch.to(device)

        optimiser.zero_grad()
        with autocast(device_type=device):
            out = model(input_batch)
            loss_val = loss_fn(out, trg_batch)
        
        if check_loss_val(
            loss_val, f'Train loss NaN @ E: {epoch + 1} B: {batch + 1}',
        ):
            running_batches += 1
            running_train_loss += loss_val.item()

            scaler.scale(loss_val).backward()
            scaler.step(optimiser)
            scaler.update()

        # compute avg loss val at the end of the epoch
        if (idx + 1) % nbatches == 0:
            avg_train_loss.append(running_train_loss / max(running_batches, 1))
            running_batches = 0
            running_train_loss = 0.0
        
            # --------- VALIDATION STEP ---------
            if valid_dl is not None:
                model.eval()
                running_valid_batches = 0
                running_valid_loss = 0.0

                with torch.no_grad():
                    for v_input, v_trg in valid_dl:
                        v_input, v_trg = v_input.to(device), v_trg.to(device)

                        with autocast(device_type=device):
                            v_out = model(v_input)
                            v_loss_val = loss_fn(v_out, v_trg)
                        
                        if check_loss_val(
                            v_loss_val, f'Valid loss NaN @ E: {epoch + 1}',
                        ):
                            running_valid_batches += 1
                            running_valid_loss += v_loss_val.item()
                    
                # store avg valid loss
                avg_v_loss_val = running_valid_loss / max(running_valid_batches, 1)
                avg_valid_loss.append(avg_v_loss_val)
                # update lr val through scheduler
                scheduler.step(avg_v_loss_val)

                model.train()

    return avg_train_loss, avg_valid_loss

In [13]:
EPOCHS: int = 10
LR: int = 1e-2
DEVICE: int = 'cuda' if torch.cuda.is_available() else 'cpu'

ae_loss = lambda x, y: nn.MSELoss()(x, y) - nn.KLDivLoss()(x, y)

params = training_setup(
    autoencoder, ae_loss, opt.Adam, opt.lr_scheduler.ReduceLROnPlateau, DEVICE,
)

avg_train_loss, avg_valid_loss = train_model(params, train_dl, EPOCHS, LR, valid_dl=valid_dl)

## Operating on device: cuda.
## Model parameters: 112457.


Training Model: 0it [00:00, ?it/s, epoch=1/10, batch=1/13, lr=1.00e-02]/home/starfloyd/anaconda3/envs/spark/lib/python3.13/site-packages/torch/nn/modules/loss.py:558: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  return F.kl_div(
Training Model: 130it [00:14,  8.87it/s, epoch=10/10, batch=13/13, lr=1.00e-02]


In [16]:
avg_train_loss, avg_valid_loss

([0.2333629417877931,
  0.16173568941079652,
  0.16095571219921112,
  0.1607849529156318,
  0.16067975530257592,
  0.1605763859473742,
  0.16049569042829367,
  0.16041228289787585,
  0.16033669618459848,
  0.16027124455341926],
 [0.15889906883239746,
  0.15558410435914993,
  0.1554948128759861,
  0.15547653287649155,
  0.15533232688903809,
  0.15527187287807465,
  0.15518460050225258,
  0.155104361474514,
  0.15503603965044022,
  0.15497250109910965])

In [15]:
assert False

AssertionError: 

In [ ]:
CONTAINER: list = []

for batch in train_dl:
    CONTAINER.append(batch)

In [ ]:
first_batch = CONTAINER[0]
x_batch, trg_batch = first_batch

idx = 0


x = x_batch[idx].unsqueeze(0)
y = autoencoder(x)

out_batch = autoencoder(x_batch)

torch.mean(torch.square(out_batch - trg_batch)), torch.std(out_batch - trg_batch)

(tensor(0.1916, grad_fn=<MeanBackward0>),
 tensor(0.3786, grad_fn=<StdBackward0>))

In [ ]:
cuda_x_batch = x_batch.to(DEVICE)
cuda_model = autoencoder.to(DEVICE)

cuda_out_batch = cuda_model(cuda_x_batch)

# torch.mean(torch.square(cuda_out_batch.detach().cpu() - trg_batch)), torch.std(cuda_out_batch.detach().cpu() - trg_batch)

nn.functional.mse_loss(cuda_out_batch, cuda_x_batch).detach().cpu()

tensor(0.1916)

In [ ]:
# def config_output_storage_hook(storage: list[Tensor] = []) -> tuple[Callable[[Any], None], list[Tensor]]:
#     """Initialises a storage hook for the model output in the feed-forward step."""
#     def hook(module: nn.Module, in_data: tuple, out_data: Tensor) -> None:
#         """Hook for dynamic forward storage for output data during training."""
#         storage.append(out_data.detach().cpu())
#         return
#     return hook, storage


# def link_hook(module: nn.Module, hook_fn: Callable):
#     """
#     Safely links and manages the lifecycle of a hook during a function call.
#     NOTE: a decorator like this is active during the whole func call, and
#           does not support ON/OFF switch.
#           This makes it unstable and prone to produce bugs when saving/loading
#           data (e.g., when saving checkpoints during training.)
#           Two other choices are: use a context manager (see `forward_data_capture`),
#           or use a state-aware `HookManager` obj, with the possibility
#           to choose to activate/deactivate the hook data capture.
#     """
#     def decorator(func: Callable):
#         @wraps(func)
#         def wrapper(*args, **kwargs) -> Any:
#             # safely attach hook
#             handle = module.register_forward_hook(hook_fn)
#             try:
#                 result = func(*args, **kwargs)
#                 return result
#             finally:
#                 # remove hook and cleanup memory
#                 handle.remove()
#                 print(f"Hook removed from {module.__class__.__name__}.")
#         return wrapper
#     return decorator


# # @errors_handler                      # catches errors from everything below
# # @link_hook(model.encoder, hook_fn)   # manages the hook
# # def train_model(*args, **kwargs):
# #     # training logic...
# #     pass

In [ ]:
# ___________________ TESTING HOOK OPs ___________________ #
assert False

def init_storage_hook() -> tuple[list, Callable]:
    """
    Initialises a storage hook for the model output in the feed-forward step.
    """
    storage: list[Any] = []

    def hook(out_data: Tensor | tuple[Tensor]) -> None:
        """Hook for dynamic forward storage during training."""
        storage.append(out_data)
    
    return storage, hook

def test_hook_working(n: int = 5) -> None:
    container, hook = init_storage_hook()
    for idx in range(n):
        t = torch.rand((10, 10))
        hook(t)
        print(
            f'At step {idx + 1}, {len(container)=}\n'
            f'Appended correctly: {torch.allclose(t, container[-1])}'
        )
    return


test_hook_working()  # ------ WORKS!



# ___________________ TESTING `errors_handler` OPs ___________________ #

@errors_handler
def handler_test(raise_err: bool) -> None:
    if raise_err:
        raise ValueError('testing decorator')
    print('executing...')
    return 1

handler_test(False)
handler_test(True)

AssertionError: 

In [ ]:
###### -------------------  TESTING TORCH OPs ------------------------ ######
assert False

def get_dataset(
    data_path: str | Path,
    batch_size: int,
    configurator: Callable[[list[str]], Dataset],
    valid_size: float | None = None,
    data_frmt: str = 'png',
    shuffle: bool = True,
) -> tuple[DataLoader, DataLoader | None]:
    """Dataset generation with given data pre-processing."""
    if valid_size and not (0 <= valid_size < 1):
        raise ValueError(f"Invalid 'valid_size' value {valid_size}, must be in [0, 1).")

    # load data files paths
    paths_list = get_data_filespaths(data_path, data_frmt, shuffle)
    # config dataset
    print('Baking the dataset...')
    dataset = configurator(paths_list)
    # split dataset in train/validation (if given `valid_size`)
    if valid_size is not None:
        v = int(valid_size * len(dataset))
        validation = Subset(dataset, torch.arange(v))
        training = Subset(dataset, torch.arange(v, len(dataset)))
        valid_dataset = DataLoader(validation, batch_size, shuffle=False)
        train_dataset = DataLoader(training, batch_size, shuffle=True)
    else:
        valid_dataset = None
        train_dataset = DataLoader(dataset, batch_size, shuffle=True)
    print('Dataset ready-to-go!')

    return train_dataset, valid_dataset


try:
    train_ds, valid_ds = map(load_dataset, (f'{BASEPATH}/train_DS.pt', f'{BASEPATH}/valid_DS.pt'))
except FileNotFoundError:
    print('No dataset(s) found :c...\n')
    train_ds, valid_ds = get_dataset(
        f'{BASEPATH}/ImgsMockDatasetDMs', BATCH_SIZE, lambda x: ImageDataset(x, PREPROCESS), VALID_SIZE,
    )
    save_dataset(train_ds, save_to=f'{BASEPATH}/train_DS.pt')
    save_dataset(valid_ds, save_to=f'{BASEPATH}/valid_DS.pt')


# ------------------------------------------------------------------------------------------------------------------- #


def np_to_torch(x: NDArray) -> Tensor:
    return F.to_tensor(x)

def pil_to_torch(img: Image) -> Tensor:
    return F.pil_to_tensor(img)


filepath: str = '/home/edoardo/Desktop/MockDataForDMs/ImgsMockDatasetDMs'
target: str = 'circle0'

img = Image.open(f'{filepath}/{target}.png').convert('RGB')
x = np.random.uniform(0, 1, (10, 10))

print(
    torch.allclose(pil_to_torch(img), ts.PILToTensor()(img)),
    torch.allclose(np_to_torch(x), ts.ToTensor()(x)),
    torch.allclose(np_to_torch(x), torch.tensor(x)),
    #torch.allclose(ts.PILToTensor()(img), ts.ToTensor()(img)),
)
img.close()

a = ['fwefwefqwef.npy', 'fwefhu2ewhf.png', 'dhqwiuedh2iuf.pt']

frmt = '*.npy'

b = frmt in a
b

AssertionError: 